In [1]:
import os
import sys
sys.path.append(os.path.join(os.path.dirname(os.getcwd()), 'src'))

from pathlib import Path
import cv2
import pickle
import numpy as np
import matplotlib.pyplot as plt
import math
import json
from tqdm import tqdm
from datetime import date
plt.rcParams["figure.figsize"] = (24,18)

from ultralytics import YOLO
from cv_utils import *

In [2]:
# YOLO model path
model_path = os.path.join(os.path.dirname(os.getcwd()), 'models')

# General data path
data_path = os.path.join(os.path.dirname(os.getcwd()), 'data')

#### Crop images to football pitch

In [3]:
for vid_name in ['real_test', 'synth_train', 'real_train']:
    print(vid_name)
    image_root_path = os.path.join(data_path, 'images/' + vid_name)
    destination_path = os.path.join(data_path, 'images/cropped_' + vid_name)
    if not os.path.exists(destination_path):
        os.mkdir(destination_path)
    for image_name in tqdm(os.listdir(image_root_path)):
        image_path = os.path.join(image_root_path, image_name)
        image = cv2.imread(image_path, cv2.COLOR_BGR2RGB)

        # Crop the image to only the pitch
        cropped_image = pitch_segment(image)

        # Write image to destination
        destination_image_path = os.path.join(destination_path, image_name)
        cv2.imwrite(destination_image_path, cropped_image)

real_test


 41%|████      | 122/300 [00:17<00:24,  7.29it/s]

#### Generate COCO dataset

In [5]:
# Generate one or multiple COCO datasets
destination_path_list = []
for vid_name in ['cropped_synth_train', 'cropped_real_train', 'cropped_real_test']:
    print(vid_name)
    dest_coco_path = export_coco_dataset_from_prediction(data_path, vid_name, 
                                        model_path=model_path, model_name="yolov8n_2nd_train.pt")
    
    destination_path_list.append(dest_coco_path)

cropped_synth_train
/media/khoa-ys/Personal/Projects/Football Analysis/Football-analysis/models


  0%|          | 0/159 [00:00<?, ?it/s]
image 1/1 /media/khoa-ys/Personal/Projects/Football Analysis/Football-analysis/data/images/cropped_synth_train/000000000005.png: 576x1024 23 persons, 5.6ms
Speed: 3.9ms preprocess, 5.6ms inference, 1.9ms postprocess per image at shape (1, 3, 576, 1024)
  1%|          | 1/159 [00:02<07:06,  2.70s/it]
image 1/1 /media/khoa-ys/Personal/Projects/Football Analysis/Football-analysis/data/images/cropped_synth_train/000000000006.png: 576x1024 23 persons, 3.2ms
Speed: 3.5ms preprocess, 3.2ms inference, 0.8ms postprocess per image at shape (1, 3, 576, 1024)
  1%|▏         | 2/159 [00:02<03:07,  1.19s/it]
image 1/1 /media/khoa-ys/Personal/Projects/Football Analysis/Football-analysis/data/images/cropped_synth_train/000000000009.png: 576x1024 24 persons, 3.8ms
Speed: 3.2ms preprocess, 3.8ms inference, 1.2ms postprocess per image at shape (1, 3, 576, 1024)
  2%|▏         | 3/159 [00:02<01:49,  1.42it/s]
image 1/1 /media/khoa-ys/Personal/Projects/Football Analy

cropped_real_train
/media/khoa-ys/Personal/Projects/Football Analysis/Football-analysis/models


  0%|          | 0/236 [00:00<?, ?it/s]
image 1/1 /media/khoa-ys/Personal/Projects/Football Analysis/Football-analysis/data/images/cropped_real_train/000000000144.png: 576x1024 18 persons, 4.1ms
Speed: 3.4ms preprocess, 4.1ms inference, 0.7ms postprocess per image at shape (1, 3, 576, 1024)
  0%|          | 1/236 [00:00<00:31,  7.45it/s]
image 1/1 /media/khoa-ys/Personal/Projects/Football Analysis/Football-analysis/data/images/cropped_real_train/000000000186.png: 576x1024 17 persons, 3.5ms
Speed: 3.0ms preprocess, 3.5ms inference, 2.0ms postprocess per image at shape (1, 3, 576, 1024)

image 1/1 /media/khoa-ys/Personal/Projects/Football Analysis/Football-analysis/data/images/cropped_real_train/000000000187.png: 576x1024 17 persons, 3.1ms
Speed: 2.9ms preprocess, 3.1ms inference, 2.6ms postprocess per image at shape (1, 3, 576, 1024)
  1%|▏         | 3/236 [00:00<00:22, 10.53it/s]
image 1/1 /media/khoa-ys/Personal/Projects/Football Analysis/Football-analysis/data/images/cropped_real_tra

cropped_real_test
/media/khoa-ys/Personal/Projects/Football Analysis/Football-analysis/models


  0%|          | 0/300 [00:00<?, ?it/s]
image 1/1 /media/khoa-ys/Personal/Projects/Football Analysis/Football-analysis/data/images/cropped_real_test/000000000047.png: 576x1024 17 persons, 3.8ms
Speed: 3.0ms preprocess, 3.8ms inference, 0.9ms postprocess per image at shape (1, 3, 576, 1024)
  0%|          | 1/300 [00:00<00:54,  5.46it/s]
image 1/1 /media/khoa-ys/Personal/Projects/Football Analysis/Football-analysis/data/images/cropped_real_test/000000000048.png: 576x1024 17 persons, 3.7ms
Speed: 3.0ms preprocess, 3.7ms inference, 0.7ms postprocess per image at shape (1, 3, 576, 1024)

image 1/1 /media/khoa-ys/Personal/Projects/Football Analysis/Football-analysis/data/images/cropped_real_test/000000000049.png: 576x1024 17 persons, 3.2ms
Speed: 3.1ms preprocess, 3.2ms inference, 1.1ms postprocess per image at shape (1, 3, 576, 1024)
  1%|          | 3/300 [00:00<00:35,  8.39it/s]
image 1/1 /media/khoa-ys/Personal/Projects/Football Analysis/Football-analysis/data/images/cropped_real_test/0

#### Merge multiple COCO datasets

In [ ]:
coco_dataset_path_1 = os.path.join(data_path, 'coco_datasets/cropped_real_train')
coco_dataset_path_2 = os.path.join(data_path, 'coco_datasets/cropped_synth_train')

dest_path=os.path.join(data_path, 'coco_datasets')

# Merge generated COCO dataset to one dataset for model training
merge_dest_path = merge_coco_dataset(destination_path_list[0], destination_path_list[1],
                   dest_path=os.path.join(data_path, 'coco_datasets'))

#### Convert COCO dataset to YOLO format

In [ ]:
# Load COCO dataset
dataset_name = 'merge_train_fixed_v2'
coco_dataset_path = os.path.join(data_path, 'coco_datasets/' + dataset_name)
test_dataset_path = os.path.join(data_path, 'coco_datasets/' + 'real_test_fixed')
# Convert COCO to YOLO
coco2yolo(coco_dataset_path, test_dataset_path=test_dataset_path)

  0%|          | 0/316 [00:00<?, ?it/s]

100%|██████████| 300/300 [00:13<00:00, 22.21it/s]


#### Augment YOLO dataset

In [ ]:
dataset_name = 'merge_train_fixed_v2_yolov8'
yolo_dataset_path = os.path.join(data_path, 'coco_datasets/' + dataset_name)
yolo_metadata_path = os.path.join(yolo_dataset_path, 'data.yaml')

augment_yolo(yolo_metadata_path, yolo_dataset_path)

/media/khoa-ys/Personal/Projects/Football Analysis/Football-analysis/data/coco_datasets/merge_train_fixed_v2_yolov8/train/images


100%|██████████| 316/316 [03:52<00:00,  1.36it/s]

2844 2844
